# News-Tweet Relevance Matching via TF-IDF and Cosine Similarity

This pipeline aims to link raw news articles with relevant social media discourse (tweets). Instead of relying on exact word overlaps, this approach uses **TF-IDF Vectorization** to map the entire vocabulary into a high-dimensional space, and then uses **Cosine Similarity** to find the closest angular distance between a tweet and an article.

**Pipeline Overview:**
1. **Data Ingestion:** Load and clean news and tweet datasets.
2. **Preprocessing:** Strip URLs, handles, and special characters to reduce noise.
3. **Vectorization:** Convert text to numerical vectors using TF-IDF (incorporating unigrams and bigrams).
4. **Similarity Matching:** Calculate Cosine Similarity to find the most relevant tweets for each specific news article.
5. **Confidence Filtering:** Apply a strict margin filter to drop generic tweets and ensure high-precision matching.

In [2]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Data Loading
First, we load the raw news articles and the dataset of tweets. We split the news file using its specific delimiter and filter out short fragments.

In [3]:
news_file = "us_newsarticle_August_1"

with open(news_file, "r", encoding="utf-8") as f:
    raw_text = f.read()

raw_articles = raw_text.split("**************")

news_articles = []

for article in raw_articles:
    article = article.strip()
    if not article:
        continue
    
    parts = article.split("-----")
    cleaned_parts = [p.strip() for p in parts if p.strip() and "August" not in p]
    
    text = " ".join(cleaned_parts)
    
    if len(text) > 50:
        news_articles.append(text)

print(f"Loaded {len(news_articles)} articles")

Loaded 27 articles


In [4]:
tweet_folder = "aug_modified1"
tweets = []

for file in sorted(os.listdir(tweet_folder)):
    path = os.path.join(tweet_folder, file)
    
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("*****")
            if len(parts) == 2:
                tweets.append(parts[1].strip())

print(f"Loaded {len(tweets)} tweets")

Loaded 169329 tweets


## 2. Text Preprocessing
To ensure accurate mathematical representations of the text, we must remove noise. This function strips URLs, user mentions (`@`), hashtags (`#`), and standardizes all text to lowercase letters.

In [5]:
def clean(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cleaned_news = [clean(a) for a in news_articles]
cleaned_tweets = [clean(t) for t in tweets]

## 3. TF-IDF Vectorization
We fit our `TfidfVectorizer` **exclusively on the news articles** to define the critical vocabulary, and then transform the tweets into this exact same dimensional space. 

* **Why?** This forces the tweets to be evaluated based purely on how much of the *news vocabulary* they utilize, ignoring slang or noise that doesn't exist in the articles.
* **N-Grams:** By setting `ngram_range=(1,2)`, the model natively understands two-word phrases (bigrams) as unique features.

In [6]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_df=0.7,
    min_df=2,
    ngram_range=(1,2),
    max_features=100000
)

news_vecs = vectorizer.fit_transform(cleaned_news)
tweet_vecs = vectorizer.transform(cleaned_tweets)

print("TF-IDF complete")
print(news_vecs.shape, tweet_vecs.shape)

TF-IDF complete
(27, 808) (169329, 808)


## 4. Article-Tweet Matching (Cosine Similarity)
We calculate the Cosine Similarity between the article vectors and the tweet vectors.

This measures the angle between vectors, accounting for the **TF-IDF weight** of the words. A tweet containing a rare, highly-weighted word will score much higher than a tweet containing several common words.

In [7]:
article_similarity = cosine_similarity(tweet_vecs, news_vecs)

tweet_article = article_similarity.argmax(axis=1)
tweet_article_score = article_similarity.max(axis=1)

print("Similarity computed")

Similarity computed


## 5. Advanced Noise Reduction (Confidence Margin)
One of the biggest issues with mapping social media text is **generic noise**. A tweet like "I can't believe this election" might match multiple political articles.

To fix this, we apply several filters:
1. **Base Threshold:** Must have a similarity score > 0.15.
2. **Length & Quality:** Drop very short tweets and explicit retweets (`RT`).
3. **Deduplication:** Ensure only unique text is processed.
4. **The Confidence Margin:** We ask: *"Does this tweet match Article A significantly better than it matches Article B?"* If the score difference between the top match and the second-best match is less than `0.1`, it is too generic and gets dropped.

In [8]:
threshold = 0.15

relevant_indices = [i for i, s in enumerate(tweet_article_score) if s > threshold]

filtered_indices = [
    i for i in relevant_indices
    if len(cleaned_tweets[i].split()) >= 6
]

non_rt_indices = [
    i for i in filtered_indices
    if not tweets[i].lower().startswith("rt")
]

unique_map = {}
for i in non_rt_indices:
    txt = tweets[i].strip()
    if txt not in unique_map:
        unique_map[txt] = i

unique_indices = list(unique_map.values())

confidence_margin = 0.1

final_indices = [
    i for i in unique_indices
    if (sorted(article_similarity[i])[-1] - sorted(article_similarity[i])[-2]) > confidence_margin
]

print(f"Final tweets: {len(final_indices)}")

Final tweets: 4182


In [9]:
sorted_indices = sorted(final_indices, key=lambda i: tweet_article_score[i], reverse=True)

seen_articles = set()
display_indices = []

for i in sorted_indices:
    art = tweet_article[i]
    
    if art not in seen_articles:
        display_indices.append(i)
        seen_articles.add(art)
    
    if len(display_indices) >= 10:
        break

print("\nTop Relevant Tweets:\n")

for idx in display_indices:
    print(f"Score: {tweet_article_score[idx]:.4f}")
    print(f"Article: {tweet_article[idx]}")
    print(f"Tweet: {tweets[idx]}")
    print("-" * 80)


Top Relevant Tweets:

Score: 0.7681
Article: 15
Tweet: i'll take fifty percent efficiency to get one hundred percent loyalty. #samuelgoldwyn #quote
--------------------------------------------------------------------------------
Score: 0.6990
Article: 22
Tweet: pope francis celebrates final mass of world youth day: pope francis bid farewell to around two million p... #wyd
--------------------------------------------------------------------------------
Score: 0.6369
Article: 9
Tweet: fbi employee acted as agent of china in exchange for prostitutes: an fbi employee of twenty years has p... #tcot
--------------------------------------------------------------------------------
Score: 0.6273
Article: 16
Tweet: gold star families call #trump's attacks on #khan family \"repugnant\" via @huffpostpol
--------------------------------------------------------------------------------
Score: 0.5737
Article: 3
Tweet: 14 cases of #zika in the florida area of the us. the fmoh ought to issue travel adv

## 6. Final Results Generation
We iterate through all 27 original articles and fetch the top 10 most relevant, high-confidence tweets for each.

In [10]:
# Setup display parameters
top_n = 10

print("-" * 80)
print("FINAL ARTICLE-TWEET RELEVANCE MATCHING")
print("-" * 80 + "\n")

# Iterate through all articles
for target_article in range(len(news_articles)):
    # Grab a snippet of the article for the header
    header_snippet = news_articles[target_article][:80].replace('\n', ' ') + "..."
    
    print(f"ARTICLE {target_article + 1}: {header_snippet}")
    print("-" * 80)
    
    # Find all tweets assigned to this specific article from our filtered list
    article_indices = [
        i for i in final_indices
        if tweet_article[i] == target_article
    ]
    
    # Sort them by similarity score descending
    article_indices = sorted(article_indices, key=lambda i: tweet_article_score[i], reverse=True)
    
    # Display the top N tweets
    if not article_indices:
        print("No relevant tweets found above the confidence threshold.\n")
    else:
        for count, idx in enumerate(article_indices[:top_n], 1):
            print(f"{count}. [{tweet_article_score[idx]:.4f}] {tweets[idx]}")
        print("\n")

--------------------------------------------------------------------------------
FINAL ARTICLE-TWEET RELEVANCE MATCHING
--------------------------------------------------------------------------------

ARTICLE 1: Melania Trump's girl-on-girl photos from racy shoot revealed Here’s the nation’s...
--------------------------------------------------------------------------------
1. [0.5261] melania at 25; hillary at 25. no contest. #trump2016
2. [0.4950] melania #trump nue : la potentielle premi\u00e8re dame voit son pass\u00e9 sulfureux resurgir
3. [0.4646] melania trump: my husband \u2018treats women the same as... #melaniatrump
4. [0.4600] i have to teach #ethics101 at trump university, melania will be in attendance. #trumpdebateexcuses
5. [0.4577] #trumpdebateexcuses melania wrote my opening speech. (and closing) \ud83d\ude06
6. [0.4031] #melania trump after liberals push gay sex for 8 years now they take the moral high ground lol
7. [0.3983] #pics #photos #photo #fotos #foto #pictures

## 6. Observations & Comparative Analysis: Cosine Similarity vs. Jaccard Index

By analyzing the outputs generated above, we can directly compare this **TF-IDF + Cosine Similarity + Confidence Margin** approach against the **Jaccard Similarity** (word overlap) approach. The core difference lies in how each method handles **Term Frequency**.

### 1. The Power of Term Frequency (TF) & Weights
* **The Jaccard Flaw (Flattened Data):** In the Jaccard approach, the algorithm extracts the top 10 words and turns them into a simple list. It completely forgets *how much* more important the #1 word is compared to the #10 word. A match on the 10th word is treated mathematically the exact same as a match on the 1st word.
* **The Cosine Advantage (Weighted Data):** This approach calculates the angle between full vectors, keeping the exact **Term Frequency-Inverse Document Frequency (TF-IDF)** weights intact. If an article mentions "Zika" 15 times (high Term Frequency) and "Zika" is rare across other articles (high IDF), its mathematical weight is massive. A tweet *must* contain "Zika" to get a high Cosine score. This is why this method perfectly isolated the Miami Zika outbreak tweets, whereas Jaccard struggled with prioritization.

### 2. Full-Spectrum Context vs. Top-10 Limitations
* **The "Warren Buffett" Problem:** In the Jaccard method, if an article about Warren Buffett endorsing Hillary Clinton had "Clinton" as the #1 word and "Buffett" as the #5 word, Jaccard simply looked for tweets containing *any* of the top 10 words. Because "Clinton" was tweeted about millions of times, the results were flooded with generic election noise.
* **The Vector Solution:** Because Cosine Similarity looks at the *entire* vocabulary space (up to 100,000 features) rather than just 10 words, it requires a tweet to align with the *proportion* of the article's words. Combined with our **Confidence Margin**, this effectively filtered out generic "Clinton" tweets that didn't specifically align with the unique "Buffett" context of the article.

### 3. Lingering Vulnerabilities: The "Says" Problem
Despite outperforming set-based matching, TF-IDF still lacks semantic comprehension. In the FBI psychic article, the model matched irrelevant tweets entirely based on the verb "says." Because the vectorizer doesn't understand parts of speech, it occasionally assigns high Term Frequency weights to common conversational filler words, leading to false positives.

## 7. Scope for Improvement and Final Conclusion

To bridge the gap between mathematical term frequency and true semantic understanding, future iterations of this pipeline should implement the following techniques:

### Areas for Optimization
1. **Named Entity Recognition (NER):** Integrating a library like `spaCy` would allow us to filter out verbs (like "says" or "happen") and focus the TF-IDF vectorizer exclusively on proper nouns, organizations, and locations. 
2. **Part-of-Speech (POS) Tagging:** By stripping out generic verbs, adverbs, and adjectives before running the vectorizer, we would prevent the model from artificially inflating the Term Frequency of conversational words.
3. **Semantic Embeddings (Word2Vec or BERT):** Moving beyond exact-word matching entirely, we could use neural network embeddings to match the *meaning* of a tweet to the *meaning* of an article, allowing us to connect texts even if they use completely different vocabularies.


# 8. Final Project Summary and Conclusion

This project implemented an NLP pipeline designed to connect traditional news reporting with real-time social media discourse. By utilizing **TF-IDF Keyword Extraction** and **Jaccard Similarity**, we analyzed how news stories propagate on Twitter and identified the strengths and limitations of token-based matching.

---

### 1. Performance Analysis: The Good, The Bad, and The Noisy

The effectiveness of the pipeline depended heavily on the **linguistic uniqueness** of the news topic:

* **The Success (High Specificity):** The **Zika Outbreak** article yielded excellent results. Because it contained specific, rare tokens like "Zika" and "CDC," the Jaccard Similarity was able to isolate highly relevant tweets with precision.
* **The Failure (Semantic Ambiguity):** The **"Prankster Cops"** article failed because its keywords—"cops" and "ice cream"—are common nouns used in thousands of unrelated contexts. The model matched the *words* but missed the *intent*, retrieving news about crime and general food tweets instead of the specific prank.
* **The Noise (Broad Context):** The **Warren Buffett/Hillary Clinton** article suffered from keyword flooding. Since "Trump" and "Clinton" were the most frequent terms on Twitter during this period, specific news about Buffett was often buried under a mountain of generic political debate.

---

### 2. Technical Evaluation

| Technique | Role | Evaluation |
| :--- | :--- | :--- |
| **TF-IDF** | Distills articles into 10 "anchor" words. | Strong at finding descriptive terms, but occasionally picks up generic high-frequency words. |
| **Jaccard Similarity** | Measures overlap between keyword sets. | Transparent and fast, but lacks "semantic depth"—it cannot distinguish between different meanings of the same word. |
| **Deduplication** | Filters out repetitive content. | Successful at removing exact retweets, ensuring a diverse list of top results. |

---

### 3. Proposed Enhancements for Future Iterations

To improve the precision of the matches, the following strategies should be considered:

1.  **Named Entity Recognition (NER):** Use libraries like `spaCy` to give higher weight to **People** (e.g., Warren Buffett) and **Organizations** (e.g., CDC) while ignoring generic nouns.
2.  **Sentence Embeddings (BERT):** Replace Jaccard Similarity with **Cosine Similarity** using BERT or RoBERTa. This would allow the model to understand that "handing out ice cream" is a positive, whimsical act, distinct from "reporting a crime."
3.  **Bigram Analysis:** Look for word pairs (e.g., "travel warning") rather than single words to preserve more context during the matching phase.

### **Final Conclusion**
This pipeline serves as a powerful baseline for information retrieval. It demonstrates that while statistical methods like TF-IDF are excellent for broad categorization, interpreting the nuance of human conversation on social media requires additional layers of contextual and semantic logic.